# Edge Live T4

Step `edge-live-t4`: 2 x T4, c = 2, 1000 test images, policies fifo, edf, sv_dpp. Select the T4 x2 accelerator.

**Inputs.** `satyapaladuguplus/scheduler-dataset-v1-simtable` (simtable_v1.jsonl, latency_v1.json) is currently PRIVATE. A run on the second account (`--account win`) needs that dataset made public first.
Codec v1 bundle: `satyapaladuguplus/scheduler-sqgan-v1` (attach once published).


In [ ]:
import os, time, warnings
os.environ.update(PYTHONWARNINGS='ignore', PYDEVD_DISABLE_FILE_VALIDATION='1', TOKENIZERS_PARALLELISM='false',
                  HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1')
warnings.filterwarnings('ignore')
T0 = time.time()

In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/kaggle/working/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
def _git(*a, cwd=None):
    return subprocess.run(["git", *a], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO + "/.git"):
    _git("fetch", "-q", "--depth", "1", "origin", BRANCH, cwd=REPO)
    _git("reset", "-q", "--hard", "FETCH_HEAD", cwd=REPO)
else:
    _git("clone", "--depth", "1", "-q", "--no-progress", "--branch", BRANCH, URL, REPO)
GIT_SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("repo @", GIT_SHA[:8], "(" + _git("log", "-1", "--format=%s", cwd=REPO) + ")")

In [ ]:
import subprocess, time
_t = time.time()
LOG = "/kaggle/working/pip_install.log"
with open(LOG, "w") as _f:
    _rc = subprocess.run(["pip", "install", "-q", "--no-input", "--disable-pip-version-check",
                          "--ignore-requires-python", "-e", ".", *['pytorch-lightning==1.9.5', 'torchmetrics==0.11.4', 'transformers==4.46.3', 'scipy', 'omegaconf', 'pytorch-msssim==1.0.0', 'lpips==0.1.4', 'PyYAML', 'einops', 'pillow']],
                         cwd=REPO, stdout=_f, stderr=subprocess.STDOUT).returncode
if _rc:
    print("".join(open(LOG).readlines()[-25:]))
    raise RuntimeError("pip install failed (rc=%s)" % _rc)
print("installed in %.0fs" % (time.time() - _t))

In [ ]:
import os
def find_input(name, root="/kaggle/input"):
    """First file called ``name`` anywhere under ``root`` (mount depth differs between kernels)."""
    for dp, _dn, fn in os.walk(root):
        if name in fn:
            return os.path.join(dp, name)
    raise FileNotFoundError(name + " not found under " + root + "; attach the dataset (public for another account)")
TABLE = find_input("simtable_v1.jsonl")
PROFILE = find_input("latency_v1.json")
print("table  :", TABLE)
print("profile:", PROFILE)

In [ ]:
import os, subprocess
OUT = "/kaggle/working/runs/" + 'edge-live-t4'
os.makedirs(OUT, exist_ok=True)
ARGV = ["python", "-u", "-m", 'Evaluation.edge.live_check', "--table", TABLE, "--profile", PROFILE, "--out", OUT + "/live_check.json",
        "--images-dir", "/kaggle/input"] + ['--n-images', '1000', '--workers', '2', '--policies', 'fifo', 'edf', 'sv_dpp']
print("$", " ".join(ARGV[3:]), flush=True)
p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
if p.wait():
    raise RuntimeError("live_check failed (rc=%d)" % p.returncode)

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-live-t4", ['--n-images', '1000', '--workers', '2', '--policies', 'fifo', 'edf', 'sv_dpp']
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "Evaluation.edge.live_check", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)